# Project 2 — Data Preparation

## Premier League players and teams (2024–25 season)

This notebook prepares the working dataset for our Project 2 analysis. It loads three raw inputs, deduplicates players who transferred mid-season, and produces a single player-level table joined with salary information and team-level context.

## Datasets used

- **`player_stats.csv`** — per-player performance statistics for the 2024–25 English Premier League season. **Observational unit:** one player.
- **`player_salaries.csv`** — per-player weekly and annual salary figures for the same season, sourced separately from the performance data. **Observational unit:** one player.
- **`team_stats.csv`** — team-level statistics for the 20 Premier League clubs in the same season. **Observational unit:** one team.

## Target merged dataset

After cleaning and merging, the observational unit will remain a single player. Each row will combine that player's individual stats, their salary, and their team's aggregate stats.

## Join strategy

1. **`player_stats` ⨝ `player_salaries`** — inner join on player name. We only want to retain players for whom we have both performance and salary data. The salary table contains more rows than the stats table, so we expect most stats-table players to find a match.
2. **(result) ⨝ `team_stats`** — left join on team. Team-level columns are attached as context to each player row.

## Expected dimensions

Roughly **~520 rows** (after deduplicating mid-season transfers and dropping unmatched players) and **~40 columns** (sum of columns from all three tables, minus redundant join keys).

## 1. Load the raw data

In [1]:
import pandas as pd

In [2]:
df_players = pd.read_csv("player_stats.csv")
df_players

,name,nation,position,team,age,born,played,starts,minutes,goals,assists,penalty_kicks,penalty_kick_attempts,yellow,red,expected_goals,progressive_carries,progressive_passes,received_progressive_passes
0,Max Aarons,England,DF,Bournemouth,25.0,2000.0,3,1,86,0,0,0,0,0,0,0.0,1,8,3
1,Joshua Acheampong,England,DF,Chelsea,19.0,2006.0,4,2,170,0,0,0,0,1,0,0.2,0,8,0
2,Tyler Adams,United States,MF,Bournemouth,26.0,1999.0,27,20,1875,0,3,0,0,7,0,1.6,13,71,10
3,Tosin Adarabioyo,England,DF,Chelsea,27.0,1997.0,21,14,1319,1,1,0,0,3,0,0.9,5,39,1
4,Simon Adingra,Cote dIvoire,"FW,MF",Brighton,23.0,2002.0,28,11,1052,2,2,0,0,0,0,2.4,50,18,128
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
566,Ashley Young,England,DF,Everton,39.0,1985.0,31,18,1785,1,3,0,0,6,1,0.3,23,89,32
567,Illia Zabarnyi,Ukraine,DF,Bournemouth,22.0,2002.0,35,34,3019,0,0,0,0,4,1,1.1,26,134,4
568,Oleksandr Zinchenko,Ukraine,"DF,MF",Arsenal,28.0,1996.0,14,4,458,0,1,0,0,1,0,0.3,10,37,12
569,Joshua Zirkzee,Netherlands,"FW,MF",Manchester Utd,24.0,2001.0,32,14,1402,3,1,0,0,2,0,4.8,14,44,69


## 2. Clean the player stats: deduplicate mid-season transfers

A player who switched clubs mid-season appears more than once in the stats table — once per team they played for. We keep the row corresponding to the team they played the most minutes for, treating that as their "primary" club for the season.

We deduplicate on the pair `(name, born)` rather than `name` alone. Two distinct players in the league could in principle share a name, and using `name` as the sole key would silently collapse them into a single row. Combining the player name with their birth year makes a name collision far less likely to also coincide with a birth-year collision, so the dedup key is much closer to a true player identifier. The `validate="1:1"` check on the salary merge below provides a second line of defense — if any same-name collision were to survive dedup, that merge would raise.

In [3]:
df_players[["name", "born"]].drop_duplicates().shape[0]  # Unique players, identified by (name, born)

559

In [4]:
df_players = df_players.sort_values('minutes', ascending=False)  # Sort by minutes played
df_players = df_players.drop_duplicates(subset=["name", "born"], keep="first")  # Keep the row with the highest minutes played per (name, born) pair
df_players.sort_index(inplace=True)  # Restore original ordering
df_players

,name,nation,position,team,age,born,played,starts,minutes,goals,assists,penalty_kicks,penalty_kick_attempts,yellow,red,expected_goals,progressive_carries,progressive_passes,received_progressive_passes
0,Max Aarons,England,DF,Bournemouth,25.0,2000.0,3,1,86,0,0,0,0,0,0,0.0,1,8,3
1,Joshua Acheampong,England,DF,Chelsea,19.0,2006.0,4,2,170,0,0,0,0,1,0,0.2,0,8,0
2,Tyler Adams,United States,MF,Bournemouth,26.0,1999.0,27,20,1875,0,3,0,0,7,0,1.6,13,71,10
3,Tosin Adarabioyo,England,DF,Chelsea,27.0,1997.0,21,14,1319,1,1,0,0,3,0,0.9,5,39,1
4,Simon Adingra,Cote dIvoire,"FW,MF",Brighton,23.0,2002.0,28,11,1052,2,2,0,0,0,0,2.4,50,18,128
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
566,Ashley Young,England,DF,Everton,39.0,1985.0,31,18,1785,1,3,0,0,6,1,0.3,23,89,32
567,Illia Zabarnyi,Ukraine,DF,Bournemouth,22.0,2002.0,35,34,3019,0,0,0,0,4,1,1.1,26,134,4
568,Oleksandr Zinchenko,Ukraine,"DF,MF",Arsenal,28.0,1996.0,14,4,458,0,1,0,0,1,0,0.3,10,37,12
569,Joshua Zirkzee,Netherlands,"FW,MF",Manchester Utd,24.0,2001.0,32,14,1402,3,1,0,0,2,0,4.8,14,44,69


In [5]:
df_salaries = pd.read_csv("player_salaries.csv")
df_salaries

,Player,Nation,Position,Team,Age,Weekly,Annual
0,Kevin De Bruyne,BEL,"MF,FW",Manchester City,33,488501,25402046
1,Erling Haaland,NOR,FW,Manchester City,24,457970,23814418
2,Casemiro,BRA,MF,Manchester Utd,32,427438,22226790
3,Mohamed Salah,EGY,FW,Liverpool,32,427438,22226790
4,Bruno Fernandes,POR,MF,Manchester Utd,29,366376,19051534
...,...,...,...,...,...,...,...
679,Cameron Humphreys,NaN,CM,Ipswich Town,21,1309,68072
680,George Abbott,NaN,DM,Tottenham,19,1007,52356
681,Rico Richards,ENG,AM,Aston Villa,21,1007,52356
682,Kobei Moore,NaN,CF,Aston Villa,20,805,41885


In [6]:
df_teams = pd.read_csv("team_stats.csv")
df_teams

,team,players,age,possession,goals,assists,penalty_kicks,penalty_kick_attempts,yellows,reds,expected_goals,expected_assists,progressive_carries,progressive_passes
0,Arsenal,25,26.6,56.8,65,53,2,2,70,6,57.6,43.2,830,1764
1,Aston Villa,28,27.8,51.0,56,45,3,6,74,3,55.7,41.5,709,1326
2,Bournemouth,29,25.9,48.1,55,39,6,7,97,3,62.4,42.6,732,1438
3,Brentford,28,26.6,47.8,64,43,5,6,61,1,57.6,41.3,579,1318
4,Brighton,32,25.7,52.0,60,40,6,6,77,3,56.5,39.7,798,1478
5,Chelsea,29,24.5,57.3,60,46,4,5,99,2,66.7,51.6,848,1573
6,Crystal Palace,29,27.0,43.1,48,37,3,4,80,4,58.7,44.7,491,1140
7,Everton,26,28.8,41.1,38,26,2,2,78,2,40.6,31.5,474,1027
8,Fulham,26,28.8,52.4,53,44,3,4,80,2,47.8,36.9,773,1524
9,Ipswich Town,32,26.6,40.5,34,25,2,2,91,5,33.6,23.7,488,896


## 3. Merge the three datasets

First we attach salaries to player stats with an inner join. Then we attach team-level context with a left join, so every salaried player keeps their team stats appended.

In [7]:
df_merged_players_stats = pd.merge(
    df_players,
    df_salaries,
    left_on=["name"],
    right_on=["Player"],
    validate="1:1",
    indicator=True,
)
df_merged_players_stats

,name,nation,position,team,age,born,played,starts,minutes,goals,...,progressive_passes,received_progressive_passes,Player,Nation,Position,Team,Age,Weekly,Annual,_merge
0,Max Aarons,England,DF,Bournemouth,25.0,2000.0,3,1,86,0,...,8,3,Max Aarons,ENG,DF,Bournemouth,24,42744,2222679,both
1,Tyler Adams,United States,MF,Bournemouth,26.0,1999.0,27,20,1875,0,...,71,10,Tyler Adams,USA,MF,Bournemouth,25,73275,3810307,both
2,Tosin Adarabioyo,England,DF,Chelsea,27.0,1997.0,21,14,1319,1,...,39,1,Tosin Adarabioyo,ENG,DF,Chelsea,26,146550,7620614,both
3,Simon Adingra,Cote dIvoire,"FW,MF",Brighton,23.0,2002.0,28,11,1052,2,...,18,128,Simon Adingra,CIV,"FW,MF",Brighton,22,15266,793814,both
4,Emmanuel Agbadou,Cote dIvoire,DF,Wolves,27.0,1997.0,15,15,1320,1,...,30,2,Emmanuel Agbadou,CIV,DF,Wolves,27,60410,3141339,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
509,Leny Yoro,France,DF,Manchester Utd,19.0,2005.0,21,12,1165,0,...,52,4,Leny Yoro,NaN,CB,Manchester Utd,19,140444,7303088,both
510,Ashley Young,England,DF,Everton,39.0,1985.0,31,18,1785,1,...,89,32,Ashley Young,ENG,DF,Everton,39,48850,2540205,both
511,Oleksandr Zinchenko,Ukraine,"DF,MF",Arsenal,28.0,1996.0,14,4,458,0,...,37,12,Oleksandr Zinchenko,UKR,"DF,MF",Arsenal,27,183188,9525767,both
512,Joshua Zirkzee,Netherlands,"FW,MF",Manchester Utd,24.0,2001.0,32,14,1402,3,...,44,69,Joshua Zirkzee,NED,"FW,MF",Manchester Utd,23,128231,6668037,both


In [8]:
df_merged_with_teams = pd.merge(
    df_merged_players_stats.drop(columns=["_merge"]),
    df_teams,
    on="team",
    how="left",
    validate="m:1",
    suffixes=("", "_team"),
    indicator=True,
)
df_merged_with_teams

,name,nation,position,team,age,born,played,starts,minutes,goals,...,assists_team,penalty_kicks_team,penalty_kick_attempts_team,yellows,reds,expected_goals_team,expected_assists,progressive_carries_team,progressive_passes_team,_merge
0,Max Aarons,England,DF,Bournemouth,25.0,2000.0,3,1,86,0,...,39,6,7,97,3,62.4,42.6,732,1438,both
1,Tyler Adams,United States,MF,Bournemouth,26.0,1999.0,27,20,1875,0,...,39,6,7,97,3,62.4,42.6,732,1438,both
2,Tosin Adarabioyo,England,DF,Chelsea,27.0,1997.0,21,14,1319,1,...,46,4,5,99,2,66.7,51.6,848,1573,both
3,Simon Adingra,Cote dIvoire,"FW,MF",Brighton,23.0,2002.0,28,11,1052,2,...,40,6,6,77,3,56.5,39.7,798,1478,both
4,Emmanuel Agbadou,Cote dIvoire,DF,Wolves,27.0,1997.0,15,15,1320,1,...,41,0,0,77,2,42.6,34.6,587,1148,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
509,Leny Yoro,France,DF,Manchester Utd,19.0,2005.0,21,12,1165,0,...,28,3,3,84,3,49.7,37.9,669,1379,both
510,Ashley Young,England,DF,Everton,39.0,1985.0,31,18,1785,1,...,26,2,2,78,2,40.6,31.5,474,1027,both
511,Oleksandr Zinchenko,Ukraine,"DF,MF",Arsenal,28.0,1996.0,14,4,458,0,...,53,2,2,70,6,57.6,43.2,830,1764,both
512,Joshua Zirkzee,Netherlands,"FW,MF",Manchester Utd,24.0,2001.0,32,14,1402,3,...,28,3,3,84,3,49.7,37.9,669,1379,both


## 4. Verify the merges

We rely on three independent checks.

**Check 1 — `validate` argument.** Each `pd.merge()` call uses `validate` (`"1:1"` for the player–salary join, `"m:1"` for the team join). Pandas raises an error if the cardinality is violated, so a successful run is itself a verification. In particular, `"1:1"` requires the player names to be unique in both `df_players` and `df_salaries`, which would catch any same-name collision that survived our `(name, born)` dedup.

**Check 2 — row counts.** After deduplicating mid-season transfers, `df_players` has one row per unique `(name, born)` pair. The first merge is an inner join, so it can only return a row count less than or equal to that number; the second merge is `m:1`, which must preserve the row count produced by the first merge.

**Check 3 — `_merge` indicator.** With `indicator=True`, every row carries a label indicating whether it matched on both sides, only the left, or only the right. For an inner join all rows must read `"both"`.

In [9]:
df_merged_players_stats.drop(columns=["_merge"]).shape

(514, 26)

The first merge returns **26 columns**, equal to the sum of the columns of `df_players` and `df_salaries` — no columns were dropped or duplicated unexpectedly.

The second merge attaches team-level context. We expect the same row count as the first merge and an additional 13 team columns.

In [10]:
df_merged_with_teams.shape

(514, 40)

Finally, we confirm via the `_merge` indicator that every row in both merges matched on both sides.

In [11]:
df_merged_players_stats["_merge"].unique()

['both']
Categories (3, str): ['left_only', 'right_only', 'both']

In [12]:
df_merged_with_teams["_merge"].unique()

['both']
Categories (3, str): ['left_only', 'right_only', 'both']

## 5. Export the merged dataset

Before saving we drop the join-key duplicates that came in from the salary table (the right-hand `Player`, `Nation`, `Position`, `Team`, `Age` columns) and the `_merge` indicator, since they're now redundant with the canonical left-hand columns.

In [13]:
df_merged_with_teams.drop(
    columns=["Player", "Nation", "Position", "Team", "Age", "_merge"],
    inplace=True,
)
df_merged_with_teams.shape

(514, 34)

In [14]:
df_merged_with_teams.to_csv("PL_players_and_teams_stats.csv", index=False)
pd.read_csv("PL_players_and_teams_stats.csv").shape  # confirm round-trip

(514, 34)